# 《LangGraph 创建智能体项目实验》

## 一、实验目的
1. 掌握 LangGraph 项目的文件结构与组织方式
2. 掌握 .env、tools.py、main.py 的创建方法
3. 掌握 langgraph.json 配置文件的作用
4. 完成 LangGraph 智能体项目的完整搭建

## 二、实验环境
- 系统：Windows 10
- Python版本：3.10
- 虚拟环境：Miniconda
- 开发工具：VS Code / Jupyter Notebook
- 依赖库：langgraph、langchain、langchain-core、langchain-deepseek、python-dotenv、loguru
- 模型：DeepSeek（deepseek-chat）

## 三、实验内容

### 3.1 创建 .env 文件

In [ ]:
LANGSMITH_TRACING="true"
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY='你注册的langsmith api key'
LANGSMITH_PROJECT='你注册的项目名称'
OPENWEATHER_API_KEY="天气助手API KEY"

### 3.2 创建 tools.py

In [ ]:
#python>=3.11
import json
import os
import httpx
import dotenv
from loguru import logger
from pydantic import Field, BaseModel
from langchain_core.tools import tool

# 加载环境变量配置
dotenv.load_dotenv()


class WeatherQuery(BaseModel):
    """
    天气查询参数模型类，用于定义天气查询工具的输入参数结构。

    :param city: 城市名称，字符串类型，表示要查询天气的城市
    """
    city: str = Field(description="城市名称")


class WriteQuery(BaseModel):
    """
    写入查询模型类

    用于定义需要写入文档的内容结构，继承自BaseModel基类

    属性:
        content (str): 需要写入文档的具体内容，包含详细的描述信息
    """
    content: str = Field(description="需要写入文档的具体内容")


@tool(args_schema=WeatherQuery)
def get_weather(city):
    """
    查询指定城市的即时天气信息。

    :param city: 必要参数，字符串类型，表示要查询天气的城市名称。
                 注意：中国城市需使用其英文名称，如 "Beijing" 表示北京。
    :return: 返回 OpenWeather API 的响应结果，URL 为
             https://api.openweathermap.org/data/2.5/weather。
             响应内容为 JSON 格式的字符串，包含详细的天气数据。
    """
    # 构建请求 URL
    url = "https://api.openweathermap.org/data/2.5/weather"

    # 设置查询参数
    params = {
        "q": city,  # 城市名称
        "appid": os.getenv("OPENWEATHER_API_KEY"),  # 从环境变量中读取 API Key
        "units": "metric",  # 使用摄氏度作为温度单位
        "lang": "zh_cn"  # 返回简体中文的天气描述
    }

    # 发送 GET 请求并获取响应
    response = httpx.get(url, params=params)

    # 将响应解析为 JSON 并序列化为字符串返回
    data = response.json()
    logger.info(f"查询天气结果：{json.dumps(data)}")
    return json.dumps(data)


@tool(args_schema=WriteQuery)
def write_file(content):
    """
    将指定内容写入本地文件

    参数:
        content (str): 要写入文件的文本内容

    返回值:
        str: 表示写入操作成功完成的提示信息
    """
    # 将内容写入res.txt文件，使用utf-8编码确保中文字符正确保存
    with open('res.txt', 'w', encoding='utf-8') as f:
        f.write(content)
        logger.info(f"已成功写入本地文件，写入内容：{content}")
        return "已成功写入本地文件。"

### 3.3 创建 main.py
编写构建图的具体逻辑，这里我们将利用预构建图API编写天气助手的代码填进去。

In [ ]:
# 导入：用LangGraph的create_react_agent
from langgraph.prebuilt import create_react_agent
import os
import dotenv
from tools import get_weather, write_file
from langchain.chat_models import init_chat_model

dotenv.load_dotenv(override=True)

# 初始化大语言模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

# 工具列表
tools = [get_weather, write_file]

# 创建智能体
agent = create_react_agent(model=llm, tools=tools)

# 测试运行
if __name__ == "__main__":
    response = agent.invoke({
        "messages": [("user", "查询北京的天气，并且把结果写入文件")]
    })

    print("最终结果：", response["messages"][-1].content)

### 3.4 创建 langgraph.json

In [ ]:
{
  "dependencies": [
    "./"
  ],
  "graphs": {
    "chatbot": "./main.py:agent"
  },
  "env": ".env"
}

## 四、实验总结

1. 掌握了 LangGraph 项目的标准文件结构
2. 理解了各文件的职责分工
3. 完成了天气助手智能体项目的完整搭建与测试